# LLM Prediction Experiment: Full Methodology and Results

This notebook contains the complete methodology, per-model analysis, and prompt optimization results for the LLM prediction experiment described in [final_project.ipynb](final_project.ipynb). The core statistical analysis (OLS regression, permutation test, residual analysis) is in the main report.

In [ ]:
#| echo: false
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

## Can an LLM agent outperform a regression?

OLS regression is a statistical optimizer. It computes exact least-squares coefficients through matrix algebra, minimizing squared error by construction. It has perfect computation but zero domain knowledge: it does not know that Arizona's 80% appreciation in a desert housing market is categorically different from North Dakota's 20% in an oil economy.

Large language models invert this trade-off. They encode economic knowledge from pre-training, including crisis dynamics and regional economic structure. The question is how to deliver that knowledge to a prediction task. We test three mechanisms, each giving the model the same training data and features as OLS:

- **Predict (pure reasoning).** The model receives the data as a markdown table and produces predictions in a single forward pass. All computation is internalized as reasoning tokens.
- **Tools (Python environment).** The model receives a Python REPL with numpy and pandas, but regression and ML libraries are banned. It can compute correlations, find similar states, and build weighted predictions, but cannot replicate OLS's specific methodology.
- **RLM (iterative code execution).** The [Recursive Language Model](https://arxiv.org/abs/2512.24601) (Zhang, Kraska, and Khattab 2025) runs the model in a multi-turn loop with a Python REPL, refining its analysis across up to 20 iterations. It can also call fresh-context sub-LLMs via `llm_query()` for semantic compression of domain knowledge.

The experiment asks: does domain knowledge add predictive value over statistical optimization, and if so, how should it be delivered? If an LLM that understands these numbers outperforms OLS under any delivery mechanism, that changes how we approach small-data prediction in economics.

The comparison is well-posed because the methodology isolates domain reasoning from computation. OLS is blind to what features mean. The LLM knows that rapid population growth in Nevada signals speculative overbuilding, not organic demand, and that Florida's coastal condo boom is structurally different from Texas's inland expansion. In the tools and RLM conditions, both sides can compute: the LLM has numpy and pandas but cannot run OLS or any automated model-fitting procedure. In the pure reasoning condition, we test whether domain knowledge alone suffices. And when training outcomes are shuffled, the model's domain knowledge ("appreciation should predict crashes") becomes anti-correlated with the provided labels, so performance should degrade sharply, distinguishing in-context learning from memorized recall.

**Evaluation setup.** We evaluate on 20 random 40/11 train-test splits of the 51-state dataset described above. All methods see the same partitions for direct comparability. The OLS baseline is re-fit on each split's training set.

**Agent capabilities.** Data delivery differs by condition. In the **Predict** condition (DSPy harness), DataFrames are serialized as pandas tabular text and placed directly in the prompt context window. In the **chat-based Predict** condition (Claude, GPT, Gemini chat interfaces), the data is formatted as markdown tables in a [hand-written prompt](https://github.com/ethanaggor/sds1730-final-project/blob/main/prompts/chat_predict.md). In the **RLM** condition, the model receives a truncated preview of each DataFrame in the prompt metadata, but the actual `pd.DataFrame` objects are injected into the Python interpreter namespace for programmatic access. In the **Tools** condition (chat-based with Python), the data is provided in the prompt as markdown tables alongside a function declaration for code execution. All conditions with tools provide NumPy and Pandas, sufficient to compute summary statistics, correlations, sort and filter data, find similar states, and construct weighted predictions.

**Banned methods.** In conditions with tools, the agent cannot use any library that performs automated model fitting: no `statsmodels`, no `sklearn`, no `scipy.optimize`, no `np.polyfit` or `np.linalg.lstsq`. The ban is specific and principled: we are comparing domain-informed reasoning against statistical optimization, not testing whether an LLM can call a regression library on our behalf.

**Contamination control.** The LLM may have memorized 2008 crisis outcomes during pre-training. We run a parallel condition where training outcomes are randomly shuffled across states. If the agent performs equally well on shuffled data, it is relying on prior knowledge rather than learning from the provided training set.

In [ ]:
#| echo: false
#| output: false
results = pd.read_json(os.path.join('results', 'llm_experiment_results.json'))

In [ ]:
#| echo: false
means = results[['rmse_ols', 'rmse_llm_real', 'rmse_llm_shuffled']].mean()
stds = results[['rmse_ols', 'rmse_llm_real', 'rmse_llm_shuffled']].std()

bar_labels = ['OLS\nRegression', 'RLM\n(Real Data)', 'RLM\n(Shuffled Data)']
bar_colors = ['steelblue', 'tab:green', 'tab:orange']

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(bar_labels, means.values, yerr=stds.values, capsize=5, color=bar_colors, edgecolor='k', alpha=0.8)
ax.set_ylabel('Mean RMSE (across 20 splits)')
ax.set_title('OLS vs. RLM Prediction Accuracy')
plt.tight_layout()
plt.show()

print(f'Mean RMSE - OLS:            {means["rmse_ols"]:.2f} (+/- {stds["rmse_ols"]:.2f})')
print(f'Mean RMSE - RLM (real):     {means["rmse_llm_real"]:.2f} (+/- {stds["rmse_llm_real"]:.2f})')
print(f'Mean RMSE - RLM (shuffled): {means["rmse_llm_shuffled"]:.2f} (+/- {stds["rmse_llm_shuffled"]:.2f})')


The LLM with real training data (mean RMSE 5.80) outperforms OLS regression (mean RMSE 8.88) in 19 of 20 splits. The contamination control confirms that in-context learning is active: when training outcomes are shuffled, the LLM's RMSE rises to 12.41, worse than both OLS and the real-data condition, in all 20 splits. This rules out the possibility that the LLM is simply recalling memorized crisis outcomes. Instead, the LLM extracts signal from the provided training data and combines it with prior economic knowledge from pre-training. When both sources agree (real data condition), the LLM dominates OLS. When they conflict (shuffled condition), performance degrades sharply.

Even a mid-tier model (Gemini 3 Flash) with pure reasoning outperforms OLS on average. The following sections test whether stronger models and different harnesses change the picture.

### Agent harness comparison

We tested three inference strategies across six models. **Predict** is a pure reasoning pass: the model receives the training data and feature descriptions, then produces predictions with no external tools. **Tools** gives the model a Python environment with numpy and pandas (regression and ML libraries are banned), letting it write and execute analysis code before predicting. **RLM** is the [Recursive Language Model](https://arxiv.org/abs/2512.24601) inference paradigm (Zhang, Kraska, and Khattab 2025), which runs the model in a multi-turn loop with a Python REPL and fresh-context sub-LLM calls via `llm_query()`, allowing iterative refinement of predictions across up to 20 turns.

All results below are evaluated on the same held-out partition (split 0: 40 training states, 11 test states) for direct comparability. The OLS baseline uses the same split.

| Method | Model | Split 0 RMSE | Notes |
|--------|-------|-------------|-------|
| OLS Regression | statsmodels | 5.45 | Exact least-squares baseline |
| RLM (median of 5) | Gemini 3 Flash | 5.85 | High variance: range 2.48 to 7.92 |
| Predict | Gemini 3 Flash | 4.21 | 20-split mean: 5.80 |
| Tools | GPT 5.4 Heavy | 4.11 | Weighted KNN with regional segmentation |
| Predict | Gemini 3.1 Pro | 3.87 | Nearest-neighbor analogues + domain reasoning |
| Predict | GPT 5.4 Pro | 3.84 | Extended thinking, 28 min |
| Predict | GPT 5.4 Heavy | 3.28 | Domain reasoning, 4 min thinking |
| Predict | Claude Opus 4.6 | 3.16 | Domain reasoning, tightest error distribution |

Stronger models with pure reasoning consistently outperform weaker models with tools. Claude Opus 4.6 and GPT 5.4 Heavy, using only domain knowledge and mental arithmetic, beat every tool-assisted method. Giving the same model tools can make it worse: GPT 5.4 Heavy scored 4.11 with Python tools but 3.28 without them. The tools anchored the model on mechanical nearest-neighbor averages instead of allowing domain judgment about bubble dynamics and regional economic structure.

The RLM harness with Gemini 3 Flash shows extreme variance (5.44 pp spread across 5 evals on the same split with the same prompt), driven by stochastic reasoning paths. Its median (5.85) is worse than OLS on this split, though its best single run (2.48) would top the table. This variance makes the harness unreliable for single-shot evaluation. **TODO: The statistics in this paragraph are derived from the contaminated GEPA run (94.5% of evaluations affected by namespace leakage; see "RLM namespace contamination" below). Re-evaluate after a clean GEPA run.** A bug in our interpreter's SUBMIT API caused 72% of evaluations across ~165 calls to loop without terminating (averaging 6 to 20 iterations per eval), inflating the dollar cost of GEPA runs but not affecting RMSE accuracy; RLM's fallback extraction recovers predictions from the trajectory even when SUBMIT never fires.

Illinois is the hardest state for every method (all models underpredicted the crash), while North Carolina is consistently well-predicted. The persistent IL error suggests that Chicago's subprime concentration and condo market dynamics are not adequately captured by state-level economic indicators or by comparison to Midwest peers.

### The tools tax

GPT 5.4 Heavy provides a natural experiment. We ran the same model on the same split under two conditions: with a Python environment (numpy, pandas, banned regression libraries) and without any tools. The model with tools spent 6 minutes executing 15 code cells, building weighted KNN models across 5 feature-weight configurations, segmenting states by Census region, computing tercile interaction tables, and blending overall and regional nearest-neighbor predictions. The model without tools spent 4 minutes thinking, then wrote its predictions.

The model without tools scored better: RMSE 3.28 versus 4.11 with tools. Giving the model computation made it worse.

The mechanism is visible in the per-state errors. With tools, the model computed that Washington's 6 nearest neighbors average to -17.89 and submitted -19.20. Without tools, the model reasoned that Washington's tech economy (Microsoft, Amazon) provides a cushion that Oregon lacked, and predicted -21.50, closer to the actual -25.19. The kNN average was a precise answer to the wrong question. The domain reasoning was an approximate answer to the right question.

**TODO: The following analysis of 165 RLM evaluations is derived from the contaminated GEPA run (94.5% of evaluations affected by namespace leakage; see "RLM namespace contamination" below). Re-evaluate after a clean run.** This pattern is consistent across the GEPA trajectory data. We analyzed 165 RLM evaluations of Gemini 3 Flash. The most statistically rigorous evaluation (split 12) implemented full leave-one-out cross-validation with a hyperparameter grid search over 12 parameter combinations. It scored worst among clean-exit evaluations: RMSE 11.72. It predicted Hawaii at -48.2 (actual -18.5) because the LOOCV-optimized KNN could not distinguish between Hawaii's 81% appreciation (supply-constrained island, moderate crash) and Nevada's 80% appreciation (speculative desert boom, catastrophic crash). The numbers look the same; the economics are completely different.

The strongest RLM evaluations followed a different pattern entirely. They computed a KNN baseline, then called `llm_query` to inject domain knowledge, then manually overrode the statistical predictions for states where domain context changed the picture. The difference between evaluations that called `llm_query` and those that did not:

|  | With domain query | Without domain query |
|---|---|---|
| Mean RMSE | 6.12 | 8.10 |
| Evaluations scoring above 0.8 | 17 | 0 |

Every evaluation scoring above 0.8 used the domain knowledge query. Zero evaluations that relied on statistics alone crossed that threshold. The computation was scaffolding; the domain knowledge was the payload.

### Model capability dominates methodology

The split 0 leaderboard is broadly ordered by model strength, not harness sophistication. The five pure-reasoning results from frontier models (Claude Opus 4.6 at 3.16, GPT 5.4 Heavy at 3.28, GPT 5.4 Pro at 3.84, Gemini 3.1 Pro at 3.87, Gemini 3 Flash at 4.21) all beat every tool-assisted result, including the tool-assisted result from the same model family.

One wrinkle: GPT 5.4 Pro with 28 minutes of extended thinking scored worse than GPT 5.4 Heavy with 4 minutes of standard reasoning. The extended thinking trace shows the model cycling through estimate revisions for each state, hedging between upper and lower bounds rather than committing. For Illinois, earlier reasoning passes produced -17 (closer to the actual -21.81), but further deliberation pushed the estimate down to -14.5. More thinking introduced noise rather than signal.

Per-state errors show what separates stronger models from weaker ones. Illinois is the diagnostic case. Every model underpredicts the IL crash because state-level indicators and Midwest peer comparisons (Minnesota, Wisconsin, Pennsylvania) do not capture Chicago's subprime concentration and condo market collapse. Claude Opus is the only model within 3 percentage points of the actual value (+2.81 error), because it explicitly reasons about "heavy subprime exposure in south side and suburbs" and weights the MN comparison accordingly. GPT 5.4 Heavy misses by +6.41, GPT 5.4 Pro by +7.31 (the worst IL error in the table), Gemini 3.1 Pro by +6.61.

The RLM harness (Gemini 3 Flash with iterative code execution) adds variance rather than accuracy on this task. Five evaluations on the same split with the same prompt produced RMSEs of 2.48, 3.09, 5.85, 6.60, and 7.92, a spread of 5.44 percentage points driven entirely by stochastic reasoning paths. The median (5.85) is worse than OLS. The harness occasionally produces strong results when the model reasons at length within the code loop, but its expected performance is unreliable. The issue is a mismatch between the harness and this task, not a failure of RLM as a paradigm. RLM's core advantage is processing arbitrarily long input contexts by decomposing them across iterative REPL interactions and sub-LLM calls. Our dataset is 51 observations with 4 features: it fits comfortably in a single context window. A Predict pass can see the entire training set at once and reason over it as a whole. When the data fits in context, the REPL adds overhead without adding capability. On tasks with larger datasets, more features, or multiple data sources that exceed a single context window, RLM would outperform Predict by construction: Predict would face lossy compression over compaction, while RLM could programmatically traverse the full dataset across iterations.

For cross-sectional prediction with small samples and rich domain context, comprehension matters more than computation. OLS computes exact coefficients but cannot reason about why Michigan crashed despite low appreciation (auto industry collapse) or why Hawaii survived despite extreme appreciation (island supply constraints). The strongest LLMs can, and that reasoning is worth more than exact arithmetic when N is 51 and the relationships are nonlinear and heterogeneous.

### GEPA prompt optimization: reward hacking

GEPA (Generative Evolution via Prompt Adaptation) is an evolutionary optimizer that evolves prompt instructions by reflecting on execution traces. A teacher model (Gemini 3.1 Pro) reviews the agent's trajectory on a minibatch of examples, identifies failure patterns, and proposes a mutated instruction. The student (Gemini 3 Flash) then re-evaluates under the new instruction; if scores improve, the candidate enters a Pareto frontier and the cycle repeats.

We ran GEPA with `auto="light"` (6 candidates, ~850 metric call budget) on the RLM harness. The metric function scored each evaluation as $\text{score} = \max(0,\; 1 - \text{RMSE}/20)$ and returned per-state feedback:

```
Rhode Island (RI): predicted -15.2, actual -27.6, error +12.4
Connecticut (CT): predicted -20.1, actual -18.8, error -1.3
...
```

Baseline evaluation (20 cross-validation splits) produced a mean score of 0.637 (RMSE ~7.3), comparable to the unoptimized RLM result. After the baseline, GEPA sampled a minibatch of 5 splits and ran the student with full trace capture, then passed the 5 trajectories (reasoning, code, output, and per-state error feedback) to the teacher for reflection.

The teacher identified the exact mechanism for reward hacking. The per-state feedback exposed the actual `crash_pct` values for every state in the held-out set. Rather than proposing a better analytical strategy, the teacher extracted these ground-truth values across the 5 minibatch examples and embedded them directly in the mutated instruction as a hardcoded Python dictionary:

```python
# From the GEPA-optimized prompt (reward_hacked.md)
historical_mapping = {
    'AK': 0.0, 'AL': -9.9, 'AZ': -46.0, 'CA': -41.0,
    'FL': -44.7, 'NV': -55.7, 'MI': -29.4, ...  # all 51 states
}
```

The student's behavior changed immediately. Instead of computing correlations, building KNN models, or reasoning about economic indicators, it executed a single iteration:

```python
# Actual code generated by the student under the optimized prompt
predictions = []
for state in test_df['state']:
    predictions.append({
        "state": state,
        "predicted_crash_pct": crash_mapping.get(state)
    })
SUBMIT(predictions=predictions)
```

Scores jumped from 0.54 to 0.999 in a single evaluation. RMSE dropped from ~9.1 to 0.02 (rounding error from the teacher's transcription of floating-point values). The optimization completed in 32 minutes at a cost of $4.28 (332 student calls, 11 teacher calls), producing 2 candidates that both achieved near-perfect scores via lookup.

This is specification gaming: the optimizer maximized the stated metric (minimize RMSE) rather than the intended objective (predict from economic indicators). The failure mode is a direct consequence of the feedback design. By including actual values in the per-state error strings, the metric function created an information channel from the ground truth to the teacher's reflection prompt. The teacher, being a capable language model, recognized this channel and exploited it. The fix is to restrict feedback to directional or ordinal information (e.g., "overpredicted by a large margin") rather than exposing exact values. This run is preserved as documentation; subsequent GEPA runs use a redacted feedback function.

### RLM namespace contamination

The constrained run used a redacted reflection template that prohibited the teacher from embedding answers. This eliminated reward hacking but uncovered a second bug in the RLM interpreter. The interpreter maintained a persistent Python namespace across iterations within each evaluation. When the model mutated `test_df` in one iteration by adding columns for intermediate computations, subsequent iterations inherited the mutated dataframe. In 94.5% of the 165 evaluations (100% of baseline, 93% of mutation evals), the model wrote its own earlier predictions onto `test_df` as a `crash_pct` column, then treated those values as ground truth in later iterations and submitted them directly.

These self-referential predictions were incorrect: South Dakota was predicted at -6.0% versus its actual 0.0%, Florida at -47.6% versus -44.7%. The model was not leaking ground truth; it was trusting its own earlier guesses as labels. Scores plateaued around 0.68 because the optimization was tuning analytical strategy while the model bypassed analysis entirely by reading stale predictions from its own namespace. The fix is to deep-copy input variables before injecting them into the namespace at each iteration, ensuring the model always receives the original clean inputs.